In [ ]:
import torch
from rdkit import Chem
import numpy as np
from rdkit.Geometry import Point3D

from mlconfgen.utils import extract_fragment
from mlconfgen.utils.mol_utils import (coord_to_pf_batched, get_moment_of_inertia_tensor_batched,
                                           prepare_masks, concat_masked_and_pad, samples_to_rdkit_mol, 
                                           )
from mlconfgen.utils import ATOM_DECODER

from mlconfgen import MLConformerGenerator, ff_inertial_fragment_matching


import torch.nn.functional as F

import py3Dmol
from IPython.display import display

from mlconfgen.cheminformatics.pipeline import set_conformer_positions
from mlconfgen.cheminformatics.shape_similarity import best_pi_rotation_by_tanimoto
from mlconfgen.conformer_generator import MLConformerGenerator
from mlconfgen.utils import (MAX_FRAG_SIZE, MIN_FRAG_SIZE, align_mol_to_principal_frame,
                    apply_transform, concat_masked_and_pad, extract_fragment,
                    get_context_shape, ifm_get_xh_from_fragment,
                    ifm_prepare_fragments_for_merge,
                    ifm_prepare_gen_fragment_context, inverse_coord_transform,
                    prepare_edm_input, samples_to_rdkit_mol,
                    split_molecule_size_constrained, standardize_mol)



def show_overlay_grid(
    reference_mol,
    candidate_mols,
    n_cols=3,
    width=300,
    height=300,
):

    def mol_to_block(mol):
        return Chem.MolToXYZBlock(mol)

    n_rows = (len(candidate_mols) + n_cols - 1) // n_cols

    view = py3Dmol.view(
        viewergrid=(n_rows, n_cols),
        width=width * n_cols,
        height=height * n_rows,
    )

    ref_block = mol_to_block(reference_mol)

    for i, cand in enumerate(candidate_mols):
        r = i // n_cols
        c = i % n_cols

        

        # Add reference (magenta)
        view.addModel(ref_block, "xyz", viewer=(r, c))
        view.setStyle(
            {"model": 0},
            {"stick": {"color": "magenta", "radius": 0.2}},
            viewer=(r, c),
        )

        #
        cand_block = mol_to_block(cand)
        view.addModel(cand_block, "xyz", viewer=(r, c))
        view.setStyle(
                {"model": 1},
                {"stick": {"radius": 0.2}},
                viewer=(r, c),
            )

        view.zoomTo(viewer=(r, c))

    return view

def compute_rigid_transform_mol(
        coord, world_coord
    ):

        centroid_coord = coord.mean(dim=1)
        centroid_world = world_coord.mean(dim=1)

        coord_centered = coord - centroid_coord[:, None, :]
        world_centered = world_coord - centroid_world[:, None, :]

        h = coord_centered.transpose(1,2) @ world_centered
        u, _, v_t = torch.linalg.svd(h, full_matrices=False)

        rotation = v_t.transpose(-2,-1) @ u.transpose(-2, -1)
        shift = centroid_world - (rotation @ centroid_coord.unsqueeze(-1)).squeeze(-1)

        return shift, rotation

def invert_transform(
        coord, shift, rotation
    ):
        return (rotation @ coord.transpose(1, 2)).transpose(1, 2) + shift[:, None, :]


def align_coord(
    cand_coord: torch.Tensor, ref_coord: torch.Tensor = None
) -> torch.Tensor:
    # move coord to center
    virtual_com = torch.mean(cand_coord, dim=0)
    cand_coord = cand_coord - virtual_com

    # Get Coords in Principal Frame
    _, aligned_coord = get_context_shape(cand_coord, include_rotation=False)

    if ref_coord is not None:
        ref_coord = ref_coord - virtual_com
        best_coord, _ = best_pi_rotation_by_tanimoto(ref_coord, aligned_coord)
        return best_coord

    return aligned_coord

def left_pad_to_right(x: torch.Tensor, pad_value: float = 0.0) -> torch.Tensor:
    """
    x: (B, N, 3)
    Assumes padding rows are on the left and equal to [pad_value, pad_value, pad_value].
    Returns tensor with non-pad rows shifted left and padding moved to the right.
    """
    B, N, D = x.shape
    out = torch.empty_like(x)

    pad_row = torch.full((D,), pad_value, dtype=x.dtype, device=x.device)

    for b in range(B):
        row = x[b]                              # (N, 3)
        is_pad = (row == pad_value).all(dim=-1)  # (N,)
        non_pad = row[~is_pad]                 # (?, 3)
        pad_len = N - non_pad.size(0)

        if pad_len > 0:
            pads = pad_row.unsqueeze(0).expand(pad_len, -1)
            out[b] = torch.cat([non_pad, pads], dim=0)
        else:
            out[b] = non_pad

    return out

def chain_merge_stack_fragments(
    ff_x: torch.Tensor,
    ff_h: torch.Tensor,
    lf_x: torch.Tensor,
    lf_h: torch.Tensor,
    ff_node_mask,
    lf_node_mask,
    
    device: torch.device,
    # max_n_atoms_final: int,
):

    n_samples = ff_node_mask.size(0)
    ff_n_atoms = ff_node_mask.size(1)
    lf_n_atoms = lf_node_mask.size(1)

    fixed_mask = torch.cat((ff_node_mask, torch.zeros_like(lf_node_mask)), dim=1).to(torch.int8)
    max_n_nodes = fixed_mask.size(1)

    # Merge Fixed Fragments X and H with Linking Fragments X and H
    merged_x = concat_masked_and_pad(
        xs=(ff_x, lf_x), masks=(ff_node_mask, lf_node_mask), pad_to=max_n_nodes
    )
    merged_h = concat_masked_and_pad((ff_h, lf_h), (ff_node_mask, lf_node_mask), pad_to=max_n_nodes)


    return merged_x, merged_h, fixed_mask

def prepare_chain_merge_input(
    n_samples: int,
    reference_context: torch.Tensor, # batched_context
    context_norms: dict,
    min_n_nodes: int,
    max_n_nodes: int,
    device: torch.device,
    pad_to: int = None,
):
   
    pad_dim = pad_to if pad_to is not None else max_n_nodes

    # Create a random list of sizes between min_n_nodes and max_n_nodes of length n_samples

    nodesxsample = torch.randint(
        min_n_nodes, max_n_nodes + 1, (n_samples,), device=device
    )

    node_mask, edge_mask = prepare_masks(
        n_nodes=nodesxsample,
        max_n_nodes=pad_dim,
        device=device,
    )

    normed_context = (
        (reference_context - context_norms["mean"]) / context_norms["mad"]
    ).to(device) * node_mask


    return (
        node_mask,
        edge_mask,
        normed_context,
    )



class IFMChainMerger:
    def __init__(self, merger, head_x, head_h, head_node_mask, n_samples, merging_steps):
        self.tail_x = head_x
        self.tail_h = head_h
        self.tail_node_mask = head_node_mask

        self.chain_x = [head_x]
        self.chain_h = [head_h]
        self.chain_node_mask = [head_node_mask]

        self.merger = merger
        self.diffusion_steps_merging = merging_steps
        self.resample_steps = 0
        self.blend_power = 3
        self.n_samples = n_samples

        self.max_n_nodes = 0 # Typically all fragment node masks will have the same size, so we can fix it here

    def build_chain(self):
        chain_masks = self.chain_node_mask
        chain_x = concat_masked_and_pad(self.chain_x, chain_masks, pad_to=None)
        chain_h = concat_masked_and_pad(self.chain_h, chain_masks, pad_to=None)
        mask = concat_masked_and_pad(chain_masks, chain_masks, pad_to=None)

        merged_mols = samples_to_rdkit_mol(chain_x, chain_h, mask, ATOM_DECODER)

        return merged_mols
        


    def attach_fragment(
                        self,
                        lf_x,
                        lf_h,
                        lf_node_mask,
                        # variance,
                       ):

        # Variance will go here
        max_n_atoms_final = int((torch.max(lf_node_mask.sum(dim=1)) + torch.max(self.tail_node_mask.sum(dim=1))).item())
        min_n_atoms_final = int((torch.min(lf_node_mask.sum(dim=1)) + torch.min(self.tail_node_mask.sum(dim=1))).item())
        
        # We Fix Tail and let link be adjusted, Fixed part stays "to the left"
        merged_x, merged_h, fixed_mask = chain_merge_stack_fragments(
            ff_x=self.tail_x,
            ff_h=self.tail_h,
            ff_node_mask = self.tail_node_mask,
            lf_x = lf_x,
            lf_h = lf_h,
            lf_node_mask = lf_node_mask,
            device=self.merger.device,
            # max_n_atoms_final=max_n_atoms_final,
        )
        
        # As both fragments were aligned to corresponding reference ones, we can set merging context to their summary MOI in PIF

        max_n_nodes = fixed_mask.size(1)


        merged_mask = concat_masked_and_pad([lf_node_mask, self.tail_node_mask], [lf_node_mask, self.tail_node_mask], pad_to=max_n_nodes)
        m_samples = samples_to_rdkit_mol(merged_x, merged_h, merged_mask, ATOM_DECODER)
        ref_mol = Chem.MolFromMolFile('./nadph.mol', sanitize=False)

        ref_mol = Chem.RemoveHs(ref_mol, sanitize=False)
        ref_context, shift, rotation, aligned_ref_coord = align_mol_to_principal_frame(
            ref_mol
        )
    
        aligned_ref_mol = set_conformer_positions(ref_mol, aligned_ref_coord)



        
        view =  show_overlay_grid(aligned_ref_mol, m_samples)
        print(f"Fragments for merging:")
        display(view)

        pf_merged_x = coord_to_pf_batched(merged_x)
        weights = torch.ones(merged_x.shape[-2], device=merged_x.device)
        moi = get_moment_of_inertia_tensor_batched(pf_merged_x, weights)  # (B,3,3)
        batch_context = moi.diagonal(dim1=-2, dim2=-1).unsqueeze(1).repeat(1, max_n_nodes, 1) # (B,N,3)

        # Stack fixed tail and flexible linkage into a single tensor, use pf_aligned coords
        z_known = torch.cat([pf_merged_x, merged_h], dim=2)

        # Prepare masks for merging
        linking_node_mask, linking_edge_mask, batch_ref_context = prepare_chain_merge_input(
            n_samples=self.n_samples,
            reference_context=batch_context,
            context_norms=self.merger.context_norms,
            min_n_nodes=min_n_atoms_final,
            max_n_nodes=max_n_atoms_final,
            device=self.merger.device,
            pad_to=max_n_nodes,
        )

        # print(f"l_node_mask: {linking_node_mask}")
        # print(f"l_edge_mask: {linking_edge_mask}")
        # print(f"fixed_mask: {fixed_mask}")
        # print(f"batch_context: {batch_ref_context}")
        # print(f"z_known: {z_known.size()}")

        # Merge linker to tail
        with torch.no_grad():
            linked_x, linked_h = self.merger.generative_model.ifm_merge_fragments_with_injection(
                linking_node_mask.to(self.merger.device),
                linking_edge_mask.to(self.merger.device),
                fixed_mask,
                context=batch_ref_context.to(self.merger.device),
                z_seed=z_known,
                diffusion_level=self.diffusion_steps_merging,
                resample_steps=self.resample_steps,
                blend_power=self.blend_power,
            )

        # Align linked fragment by matching the fixed fragment with the tail
        # We have constant number of atoms in tail
        tail_n_atoms = int(torch.max(self.tail_node_mask.sum(dim=1)).item())
            
        cur_tail_x = torch.split(linked_x * fixed_mask, int(max_n_nodes / 2), dim=1)[0]
        shift, rotation = compute_rigid_transform_mol(cur_tail_x, self.tail_x)

        linked_x = invert_transform(linked_x, shift, rotation)



        f_samples = samples_to_rdkit_mol(linked_x, linked_h, merged_mask, ATOM_DECODER)
        view =  show_overlay_grid(aligned_ref_mol, f_samples)
        print(f"Fragments after merging:")
        display(view)
        

        notfixed_mask = 1 - fixed_mask


        # And our link becomes new tail
        # We can apply split after we move all zeros from fixed mask to the right
        self.tail_node_mask = torch.split(left_pad_to_right(linking_node_mask * notfixed_mask), int(max_n_nodes / 2), dim=1)[0]
        self.tail_x = torch.split(left_pad_to_right(linked_x * notfixed_mask), int(max_n_nodes / 2), dim=1)[0]
        self.tail_h = torch.split(left_pad_to_right(linked_h * notfixed_mask), int(max_n_nodes / 2), dim=1)[0]

        # Now linked fragment should go into total chain
        self.chain_x.append(self.tail_x)
        self.chain_h.append(self.tail_h)
        self.chain_node_mask.append(self.tail_node_mask)

        # And merge should be complete




In [ ]:
import torch
from rdkit import Chem

from mlconfgen.cheminformatics.pipeline import set_conformer_positions
from mlconfgen.cheminformatics.shape_similarity import best_pi_rotation_by_tanimoto
from mlconfgen.conformer_generator import MLConformerGenerator
from mlconfgen.utils import (MAX_FRAG_SIZE, MIN_FRAG_SIZE, align_mol_to_principal_frame,
                    apply_transform, concat_masked_and_pad, extract_fragment,
                    get_context_shape, ifm_get_xh_from_fragment,
                    ifm_prepare_fragments_for_merge,
                    ifm_prepare_gen_fragment_context, inverse_coord_transform,
                    prepare_edm_input, samples_to_rdkit_mol,
                    split_molecule_size_constrained, standardize_mol)


def ifm_chain_merging(
    reference_conformer: Chem.Mol,
    n_samples: int,
    generator: MLConformerGenerator,
    merger: MLConformerGenerator = None,
    variance: int = 1,
    n_atoms: int = None,
    resample_steps: int = 0,
    diffusion_steps_merging: int = 10,
    min_frag_size: int = MIN_FRAG_SIZE,
    max_frag_size: int = MAX_FRAG_SIZE,
    max_iter: int = 200,
    verbose: bool = False,
    predict_bonds: bool = False,
    optimize_geometry: bool = False,
) -> list[Chem.Mol]:
    """
    """

    if merger is None:
        merger = generator

    g_device = generator.device
    m_device = merger.device

    g_context_norms = generator.context_norms
    m_context_norms = merger.context_norms

    # Strip of Hs and align Reference to principal Inertial Frame, saving rotation and shift
    ref_mol = Chem.RemoveHs(reference_conformer, sanitize=False)
    ref_context, shift, rotation, aligned_ref_coord = align_mol_to_principal_frame(
        ref_mol
    )

    if n_atoms is None:
        n_nodes = aligned_ref_coord.size(0)
    else:
        n_nodes = n_atoms

    min_n_atoms_final = n_nodes - variance
    max_n_atoms_final = n_nodes + variance

    aligned_ref_mol = set_conformer_positions(ref_mol, aligned_ref_coord)

    # Split Reference molecule into fragments
    fragment_sets = split_molecule_size_constrained(
        mol=aligned_ref_mol,
        min_size=min_frag_size,
        max_size=max_frag_size,
        max_iter=max_iter,
        verbose=verbose,
    )

    if len(fragment_sets) == 0:
        raise RuntimeError(
            "Could not split reference molecule into fragments, aborting IFM generation."
        )

    # Extract fragments as individual conformers
    extracted_frags = []

    for frag_set in fragment_sets:
        extracted_frags.append(extract_fragment(aligned_ref_mol, frag_set))

    fragment_contexts = []
    fragment_shifts = []
    fragment_rotations = []
    ref_fragment_coords = []

    edm_inputs = []

    # Determine the max number of nodes
    max_n_nodes = 0
    for frag in extracted_frags:
        _n_atoms = frag.GetNumHeavyAtoms()
        if _n_atoms > max_n_nodes:
            max_n_nodes = _n_atoms

    # Align Fragments to their respective Principal Inertial Frames,
    # while remembering corresponding Shifts and Rotations
    # Prepare concatenate-able edm inputs for all fragments
    for frag in extracted_frags:
        f_n_atoms = frag.GetNumHeavyAtoms()
        f_context, f_shift, f_rotation, f_coord = align_mol_to_principal_frame(frag)
        fragment_contexts.append(f_context)
        fragment_shifts.append(f_shift)
        fragment_rotations.append(f_rotation)
        ref_fragment_coords.append(f_coord)

        node_mask, edge_mask, batch_context = prepare_edm_input(
            n_samples=n_samples,
            reference_context=f_context,
            context_norms=g_context_norms,
            min_n_nodes=f_n_atoms,
            max_n_nodes=f_n_atoms,
            device=g_device,
            pad_to=max_n_nodes,
        )

        edm_inputs.append(
            {
                "node_mask": node_mask,
                "edge_mask": edge_mask,
                "batch_context": batch_context,
            }
        )

    total_node_mask = torch.cat([x["node_mask"] for x in edm_inputs], 0)
    total_batched_context = torch.cat([x["batch_context"] for x in edm_inputs])

    # Total Edge mask is a bit trickier:
    helper_node_mask = total_node_mask.clone().squeeze()
    batch_size = helper_node_mask.size(0)
    max_n_nodes = helper_node_mask.size(1)

    # Compute Total Edge Mask
    total_edge_mask = helper_node_mask.unsqueeze(1) * helper_node_mask.unsqueeze(2)
    diag_mask = ~torch.eye(
        total_edge_mask.size(1), dtype=torch.bool, device=g_device
    ).unsqueeze(0)
    total_edge_mask *= diag_mask
    total_edge_mask = total_edge_mask.view(batch_size * max_n_nodes * max_n_nodes, 1)

    # Generating samples matching all isolated fragments at the same time
    print("Generating fragments ...")
    with torch.no_grad():
        total_x, total_h = generator.generative_model(
            total_node_mask,
            total_edge_mask,
            total_batched_context,
            resample_steps=resample_steps,
        )

    # Split generated tensor into samples generated per fragment
    x_fragments = torch.split(total_x, n_samples, dim=0)
    h_fragments = torch.split(total_h, n_samples, dim=0)
    node_masks = torch.split(total_node_mask, n_samples, 0)

    # Batchify shifts and rotations
    # Apply corresponding inverse coord transforms to each fragment coordinates
    coord_for_merge = []

    for i, frag_coord in enumerate(x_fragments):
        batch_shift = fragment_shifts[i].unsqueeze(0).expand(n_samples, -1).to(m_device)
        batch_rot = (
            fragment_rotations[i]
            .transpose(0, 1)
            .unsqueeze(0)
            .expand(n_samples, 3, -1)
            .to(m_device)
        )

        # Align generated fragments to principal frame to maximize ref fragment volume overlay
        aligned_x = []
        frag_coord = frag_coord.to("cpu")
        for old_x in frag_coord:
            try:
                aligned_x.append(
                    align_coord(cand_coord=old_x, ref_coord=ref_fragment_coords[i])
                )
            except RuntimeError:
                pass

        aligned_x = torch.stack(aligned_x, dim=0).to(m_device)

        new_x = inverse_coord_transform(
            coord=aligned_x,
            shift=batch_shift,
            rotation=batch_rot.transpose(1, 2),
        )

        # Save prepared coordinates for Merging the fragments
        coord_for_merge.append(new_x)

    # Doing chain merging
    # Instantiate the merger with the first fragment doing left -> right
    print("All Generated and aligned, Starting chain merging")

    # Need to find an algorithm to order fragments in chain merging order
    def _fix(tup):
        rev_tup = tup[::-1]
        fixed_tup = list(rev_tup)
        fixed_tup[2] = rev_tup[3]
        fixed_tup[3] = rev_tup[2]
        return tuple(fixed_tup)

    coord_for_merge = _fix(coord_for_merge)
    h_fragments = _fix(h_fragments)
    node_masks = _fix(node_masks)
    
    
    chain_merger = IFMChainMerger(
                                    merger=merger,
                                    head_x=coord_for_merge[0],
                                    head_h=h_fragments[0],
                                    head_node_mask=node_masks[0],
                                    n_samples=n_samples,
                                    merging_steps=diffusion_steps_merging,
                                 )

    n_f = len(coord_for_merge) - 1
    

    for i in range(n_f):
        idx = i + 1
        print(f"Attaching fragment {idx} out of {n_f}")
        chain_merger.attach_fragment(lf_x=coord_for_merge[idx], lf_h=h_fragments[idx], lf_node_mask=node_masks[idx])

    print("Woah, Chain Merging complete...")

    chain = chain_merger.build_chain()

    display(show_overlay_grid(aligned_ref_mol, chain))

    return chain

    
        


In [ ]:
import time
import torch

from rdkit import Chem, RDLogger
from rdkit.Chem import Draw

from mlconfgen import MLConformerGenerator, evaluate_samples



RDLogger.DisableLog('rdApp.*')

if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")

print(f"Intitialising model on {device}")
    
generator = MLConformerGenerator(
                                 edm_weights="../../edm_moi_chembl_6_39_fragments.pt",
                                 adj_mat_seer_weights="../../adj_mat_seer_chembl_15_39.pt",
                                 device=device,
                                 diffusion_steps=20,
                                )

merger = MLConformerGenerator(
                                 edm_weights="../../edm_moi_chembl_15_39.pt",
                                 adj_mat_seer_weights="../../adj_mat_seer_chembl_15_39.pt",
                                 device=device,
                                 diffusion_steps=100,
                                )


# Load a Reference conformer
ref_mol = Chem.MolFromMolFile('./nadph.mol', sanitize=False)

# Generate Samples
print("IFM Generation started...")
start = time.time()

N_SAMPLES = 10

samples = ifm_chain_merging(   
                                     reference_conformer=ref_mol, 
                                     n_samples=N_SAMPLES,  
                                     generator=generator,
                                     merger=merger,
                                     variance=1,
                                     predict_bonds=False,
                                     verbose=True,
                                     diffusion_steps_merging=10,
                                    )
print(f"IFM Generation complete in {round(time.time() - start, 2)}")

# show_overlay_grid(ref_mol, samples)

